In [1]:
import os

In [2]:
%pwd

'f:\\nlp\\Text-Summarizer-Project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'f:\\nlp\\Text-Summarizer-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [7]:
from textSummarizer.constants import * 
from textSummarizer.utils.common import read_yaml, create_directories


In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):
        self.config_filepath = Path(config_filepath)
        self.params_filepath = Path(params_filepath)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root], verbose=True)
    
    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir], verbose=True)
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name
        )
        return model_evaluation_config


In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
from evaluate import load as load_metric
import torch
import pandas as pd
from tqdm import tqdm


In [10]:
class ModelEvaluation:
    def __init__(self,config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self,list_of_elements, batch_size):
        """ split the dataset into smaller batches that we can process simultaneously
        yield successive batch-sized chunks from list_of_elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer,
                                    batch_size=8, device='cpu',
                                    column_text="article", column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(zip(article_batches, target_batches),
                                                total=len(article_batches)):
            inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                               padding="max_length", return_tensors="pt")

            inputs = {k: v.to(device) for k, v in inputs.items()}

            summaries = model.generate(input_ids=inputs["input_ids"],
                                       attention_mask=inputs["attention_mask"],
                                       length_penalty=0.8, num_beams=4, max_length=128)

            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                for s in summaries
            ]
            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        score = metric.compute()
        return score
    
    def evaluate(self):
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path)
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)

        # loading data
        dataset = load_from_disk(self.config.data_path)
        rough_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_metric = load_metric("rouge")
        score = self.calculate_metric_on_test_ds(
            dataset['test'], rouge_metric, model, tokenizer,batch_size=2,column_text='dialogue',column_summary='summary')
        
        rouge_dict = {rn: score[rn] for rn in rough_names}
        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        df.to_csv(self.config.metric_file_name, index=False)




In [11]:
try:
    config_manager = ConfigurationManager()
    model_evaluation_config = config_manager.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(model_evaluation_config)
    model_evaluation.evaluate()
except Exception as e:
    raise e  

[2025-07-31 01:07:23,324:INFO:yaml file: config\config.yaml loaded sucessfully]
[2025-07-31 01:07:23,326:INFO:yaml file: params.yaml loaded sucessfully]
[2025-07-31 01:07:23,328:INFO:created directory at: artifacts]
[2025-07-31 01:07:23,329:INFO:created directory at: artifacts/model_evaluation]


100%|██████████| 410/410 [45:17<00:00,  6.63s/it]

[2025-07-31 01:52:45,089:INFO:Using default tokenizer.]
